In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
DATA_DIR = Path("../data/raw_data")
if os.path.isdir(DATA_DIR):
    print("Directory exists")
else:
    os.makedirs(DATA_DIR)
    print("Directory created")

Directory exists


In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

In [4]:
train.shape, test.shape

((668665, 15), (286571, 14))

In [6]:
train.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [7]:
target_map = {'No': 0, 'Yes': 1}
y = train["Will_Buy_EV"].map(target_map)

In [8]:
X = train.drop(columns=["Will_Buy_EV", "id"], errors='ignore')
X_test = test.drop(columns=["id"], errors='ignore')

In [10]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline

In [9]:
def get_preprocessor():
    # Preprocessor for XGBoost and LightGBM
    number_transformer = Pipeline(steps=[("scaler", StandardScaler())])
    category_transformer = Pipeline(steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", number_transformer, make_column_selector(dtype_include='number')),
            ("categorical", category_transformer, make_column_selector(dtype_include='object'))
        ],
        remainder='drop'
    )
    return preprocessor